# Install Java and Spark on Hadoop

In [1]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
os.environ["SPARK_HOME"] = "/home/xn/spark"

# Create a SparkSession in Python

In [2]:
# start pyspark
%pip install findspark
import findspark
findspark.init()

Note: you may need to restart the kernel to use updated packages.


In [3]:
import pyspark
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local")\
          .appName("Spark APIs Exercises")\
          .config("spark.some.config.option", "some-value")\
          .getOrCreate()

your 131072x1 screen size is bogus. expect trouble
25/04/23 06:28:48 WARN Utils: Your hostname, xn resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
25/04/23 06:28:48 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/04/23 06:28:49 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
%pip install shapely

Note: you may need to restart the kernel to use updated packages.


In [35]:
from pyspark.sql import SparkSession
from shapely.geometry import Polygon
from pyspark.sql.functions import col
from pyspark.sql.types import IntegerType, StructType, StructField, DoubleType, StringType

# Load Parquet file
df = spark.read.parquet("shapes.parquet")
df.show()

+--------+--------------------+
|shape_id|            vertices|
+--------+--------------------+
| Shape_0|[[35, 28], [43, 2...|
| Shape_1|[[44, 16], [53, 1...|
| Shape_2|[[67, 84], [76, 8...|
| Shape_3|[[29, 37], [37, 3...|
| Shape_4|[[51, 79], [54, 7...|
| Shape_5|[[55, 33], [62, 3...|
| Shape_6|[[17, 88], [21, 8...|
| Shape_7|[[1, 2], [10, 2],...|
| Shape_8|[[65, 14], [69, 1...|
| Shape_9|[[12, 48], [19, 4...|
|Shape_10|[[97, 20], [100, ...|
|Shape_11|[[7, 38], [15, 38...|
|Shape_12|[[69, 65], [78, 6...|
|Shape_13|[[35, 8], [42, 8]...|
|Shape_14|[[96, 35], [100, ...|
|Shape_15|[[74, 62], [81, 6...|
|Shape_16|[[75, 97], [80, 9...|
|Shape_17|[[12, 4], [19, 4]...|
|Shape_18|[[54, 17], [57, 1...|
|Shape_19|[[100, 66], [106,...|
+--------+--------------------+
only showing top 20 rows



# Example 1: WordCount with Spark DataFrames and Spark RDDs




In [4]:
# Load the data
!git clone https://github.com/nnthaofit/CSC14118.git

fatal: destination path 'CSC14118' already exists and is not an empty directory.


### Spark DataFrame-based WordCount

In [5]:
linesDF = spark.read.text("CSC14118/ppap.txt")
linesDF.show(linesDF.count(),truncate = False)

from pyspark.sql import functions as f
wordsDF = linesDF.withColumn("word", f.explode(f.split(f.col("value"), " ")))\
    .groupBy("word")\
    .count()\
    .sort("count", ascending = False)
wordsDF.show()

+----------------------------+
|value                       |
+----------------------------+
|ppap                        |
|i have a pen                |
|i have an apple             |
|ah apple pen                |
|i have a pen                |
|i have a pineapple          |
|ah pineapple pen            |
|ppap pen pineapple apple pen|
+----------------------------+

+---------+-----+
|     word|count|
+---------+-----+
|      pen|    6|
|     have|    4|
|        i|    4|
|    apple|    3|
|pineapple|    3|
|        a|    3|
|     ppap|    2|
|       ah|    2|
|       an|    1|
+---------+-----+



###RDD-based WordCount

In [6]:
linesRdd = spark.sparkContext.textFile("CSC14118/ppap.txt")
wordsRdd = linesRdd.flatMap(lambda line: line.split(" ")) \
    .map(lambda word: (word, 1)) \
    .reduceByKey(lambda a, b: a + b)\
    .sortBy(lambda pair:-1*pair[1])
wordsRdd.collect()

[('pen', 6),
 ('i', 4),
 ('have', 4),
 ('a', 3),
 ('apple', 3),
 ('pineapple', 3),
 ('ppap', 2),
 ('ah', 2),
 ('an', 1)]

# Exercise 1: Data query with Spark DataFrame

In [7]:
# clone the example data files from GitHub to Drive
!git clone https://github.com/nnthaofit/CSC14118.git

fatal: destination path 'CSC14118' already exists and is not an empty directory.


###0. Load the data file: movies.json

In [6]:
movieDF = spark.read.json("CSC14118/movies.json")
movieDF.show()

+----------------+--------------------+--------------------+----+
|            cast|              genres|               title|year|
+----------------+--------------------+--------------------+----+
|              []|                  []|After Dark in Cen...|1900|
|              []|                  []|Boarding School G...|1900|
|              []|                  []|Buffalo Bill's Wi...|1900|
|              []|                  []|              Caught|1900|
|              []|                  []|Clowns Spinning Hats|1900|
|              []|[Short, Documentary]|Capture of Boer B...|1900|
|              []|                  []|The Enchanted Dra...|1900|
|   [Paul Boyton]|                  []|   Feeding Sea Lions|1900|
|              []|            [Comedy]|How to Make a Fat...|1900|
|              []|                  []|     New Life Rescue|1900|
|              []|                  []|    New Morning Bath|1900|
|              []|                  []|Searching Ruins o...|1900|
|         

### 1a. Show the schema of DataFrame that stores the movies dataset.

In [9]:
movieDF.printSchema()

root
 |-- cast: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- genres: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- title: string (nullable = true)
 |-- year: long (nullable = true)



### 1b. Show the number of distinct movies in the dataset

In [10]:
from pyspark.sql.functions import count, countDistinct

movieDF.select(count("title")).show()

+------------+
|count(title)|
+------------+
|       28795|
+------------+



### 2. Count the number of movies released during the years 2012 and 2015 (included)

In [ ]:
movieDF.filter((movieDF.year >= 2012) & (movieDF.year <= 2015)).count()

1015

### 3. Show the year in which the number of movies released is highest. One highest year is enough

In [8]:
from pyspark.sql import functions as f

yearDF = movieDF.groupBy("year").count().sort('count', ascending = False).limit(1)
yearDF.show()


+----+-----+
|year|count|
+----+-----+
|1919|  634|
+----+-----+



### 4. Show the list of movies such that for each film, the number of actors/actresses is at least five, and the number of genres it belongs to is at most two genres.

In [13]:
# from pyspark.sql.functions import expr

# castDF = movieDF.withColumn("caster", f.explode(movieDF.cast)).groupBy(movieDF.title).count().withColumnRenamed('count', 'castCount')
# castDF = castDF.filter(castDF["castCount"] >= 5)
# genreDF = movieDF.withColumn("gen", f.explode(movieDF.genres)).groupBy(movieDF.title).count().withColumnRenamed('count', 'genreCount')
# genreDF = genreDF.filter(genreDF["genreCount"] >= 2)
# castgenreDF = castDF.join(genreDF, castDF.title == genreDF.title, 'inner').select(castDF.title).show()

resDF = movieDF.where(f.size(movieDF.cast) >= 5).where(f.size(movieDF.genres) >= 2).show()


+--------------------+--------------------+--------------------+----+
|                cast|              genres|               title|year|
+--------------------+--------------------+--------------------+----+
|[Lon Chaney, Leat...|      [Crime, Drama]|   The Ace of Hearts|1921|
|[Madge Kennedy, M...|     [Comedy, Drama]|  The Purple Highway|1923|
|[Laura La Plante,...|[Romance, Comedy,...|          The Teaser|1925|
|[Fredric March, O...|    [Drama, Romance]|     Anthony Adverse|1936|
|[Gary Cooper, Mad...|  [Drama, Adventure]|The General Died ...|1936|
|[William Powell, ...|[Musical, Biography]|  The Great Ziegfeld|1936|
|[Gary Cooper, (, ...|   [Comedy, Romance]|Mr. Deeds Goes to...|1936|
|[William Powell, ...|   [Comedy, Romance]|      My Man Godfrey|1936|
|[Stuart Erwin, (,...|   [Comedy, Musical]|      Pigskin Parade|1936|
|[Clark Gable, Jea...|  [Drama, Adventure]|       San Francisco|1936|
|[Tyrone Power, My...|    [Romance, Drama]|      The Rains Came|1939|
|[Judy Garland, Fr..

### 5. Show the **movies** whose names are longest

In [ ]:
# longName = movieDF.select(f.max(f.length(movieDF.title))).collect()[0][0]
# movieDF.filter(f.length(movieDF.title) == longName).show()
longname = movieDF.select(f.max(f.length(movieDF.title))).collect()


### 6. Show the movies whose name contains the word “fighting” (case-insensitive).

In [15]:
# Show movies whose name have the word "fighting"
movieDF.filter(f.lower(movieDF.title).contains("fighting")).show()

+--------------------+---------------+--------------------+----+
|                cast|         genres|               title|year|
+--------------------+---------------+--------------------+----+
|[Bessie Love, Ann...|[Comedy, Drama]|  A Fighting Colleen|1919|
|[Blanche Sweet, R...|      [Western]|     Fighting Cressy|1919|
|[Harry T. Morey, ...|        [Drama]|    Fighting Destiny|1919|
|[Tom Mix, Teddy S...|      [Western]|   Fighting for Gold|1919|
|[Jack Perrin, Hoo...|      [Western]|  The Fighting Heart|1919|
|[Art Acord, Mildr...|      [Western]|   The Fighting Line|1919|
|[William Duncan, ...|       [Action]|  The Fighting Guide|1922|
|[Tom Mix, Patsy R...|      [Western]| The Fighting Streak|1922|
|[Richard Barthelm...|   [Historical]|  The Fighting Blade|1923|
|[Ernest Torrence,...|       [Comedy]| The Fighting Coward|1924|
|[Jack Hoxie, Hele...|      [Western]|       Fighting Fury|1924|
|[Pat O'Malley, Ma...|        [Drama]|The Fighting Adve...|1924|
|[Fred Thomson, Ha...|   

### 7. Show the list of distinct genres appearing in the dataset

In [16]:
movieDF.select(f.explode(movieDF.genres).alias("genre")).distinct().show()

+-------------+
|        genre|
+-------------+
|        Crime|
|      Romance|
|     Thriller|
|      Slasher|
|Found Footage|
|    Adventure|
|         Teen|
| Martial Arts|
|       Sports|
|        Drama|
|          War|
|  Documentary|
|       Family|
|      Fantasy|
|       Silent|
|     Disaster|
|        Legal|
|      Mystery|
| Supernatural|
|     Suspense|
+-------------+
only showing top 20 rows



### 8. List all movies in which the actor Harrison Ford has participated.

In [17]:
exploded_cast = movieDF.withColumn("actor", f.explode(movieDF.cast))
exploded_cast.filter(f.lower(exploded_cast.actor) == "harrison ford").select(exploded_cast.title).show()

+--------------------+
|               title|
+--------------------+
|Experimental Marr...|
| Happiness a la Mode|
|Romance and Arabella|
|      The Third Kiss|
|The Veiled Adventure|
|          Who Cares?|
|You Never Saw Suc...|
| The Wonderful Thing|
|      Find the Woman|
| The Primitive Lover|
|     Smilin' Through|
|     When Love Comes|
| Little Old New York|
|     Three Miles Out|
|           The Wheel|
|       Almost a Lady|
| Hell's Four Hundred|
|   The Nervous Wreck|
|  Up in Mabel's Room|
|         Golf Widows|
+--------------------+
only showing top 20 rows



### 9. List all movies in which the actors/actresses whose names include the word “Lewis“ (case-insensitive) have participated.

In [18]:
exploded_cast = movieDF.withColumn("actor", f.explode(movieDF.cast))
lewis_cast = exploded_cast.filter(f.lower(exploded_cast.actor).contains("lewis"))
lewis_cast.select(lewis_cast.title).distinct().show()

+--------------------+
|               title|
+--------------------+
| Inez from Hollywood|
|The Ballad of Jac...|
|             Salvage|
|At War with the Army|
|  Sex and the City 2|
|   Diary of a Madman|
|      Going Straight|
|         Cinderfella|
|   Gangs of New York|
|       That's My Boy|
|     Rock-A-Bye Baby|
|        The Crucible|
|The Million Dolla...|
|The Girl from Mon...|
|             Romance|
|  New Morals for Old|
|The Hardys Ride High|
|Andy Hardy Meets ...|
|Love Laughs at An...|
|    Cheaper to Marry|
+--------------------+
only showing top 20 rows



### 10. Show top five actors/actresses that have participated in most movies.

In [19]:
exploded_cast = movieDF.withColumn("actor", f.explode(movieDF.cast))
exploded_cast.groupBy(exploded_cast.actor).count().sort('count', ascending = False).show(5)

+----------------+-----+
|           actor|count|
+----------------+-----+
|    Harold Lloyd|  190|
|     Hoot Gibson|  142|
|      John Wayne|  136|
|Charles Starrett|  116|
|    Bebe Daniels|  103|
+----------------+-----+
only showing top 5 rows



#Exercise 2: RDD-based mainpulation


*   The data is already in one ore more RDDs.
*   You must not convert RDD to DF or use pure Python code.


### 1. Consider a string s that includes only alphabetical letters and spaces. Check whether s is a palindrome (case-insensitive).

In [20]:
s = "racecar"
rdd = spark.sparkContext.parallelize(s.lower(), 1).filter(lambda letter: letter != ' ')
rdd.collect()



['r', 'a', 'c', 'e', 'c', 'a', 'r']

In [21]:
rdd_id = spark.sparkContext.parallelize(range(0, rdd.count()), 1)
rdd_id.collect()
rdd_pair = rdd.zip(rdd_id)
rdd_pair.collect()

[('r', 0), ('a', 1), ('c', 2), ('e', 3), ('c', 4), ('a', 5), ('r', 6)]

In [22]:
rddb = rdd_pair.sortBy(lambda row: row[1]*-1)
rddb.collect()

[('r', 6), ('a', 5), ('c', 4), ('e', 3), ('c', 2), ('a', 1), ('r', 0)]

In [23]:
rdd_cmp = rdd_pair.zip(rddb)
rdd_cmp.collect()

[(('r', 0), ('r', 6)),
 (('a', 1), ('a', 5)),
 (('c', 2), ('c', 4)),
 (('e', 3), ('e', 3)),
 (('c', 4), ('c', 2)),
 (('a', 5), ('a', 1)),
 (('r', 6), ('r', 0))]

In [24]:
if (rdd_cmp.filter(lambda row: row[0][0]!=row[1][0]).count() == 0):
  print("Palindrome")
else:
  print("Not Palindrome")

Palindrome


### 2. Consider a string s that includes only alphabetical letters and spaces. Check whether s is a pangram (case-insensitive).

In [25]:
s = "The quick brown fox jumps over the lazy dog"

rdd = spark.sparkContext.parallelize(s.lower(), 1).filter(lambda letter: letter != ' ').sortBy(lambda letter: ord(letter))


rdd.collect()


['a',
 'b',
 'c',
 'd',
 'e',
 'e',
 'e',
 'f',
 'g',
 'h',
 'h',
 'i',
 'j',
 'k',
 'l',
 'm',
 'n',
 'o',
 'o',
 'o',
 'o',
 'p',
 'q',
 'r',
 'r',
 's',
 't',
 't',
 'u',
 'u',
 'v',
 'w',
 'x',
 'y',
 'z']

In [26]:
if rdd.distinct().count() == 26:
  print("Anagram")
else:
  print("Not anagram")

Anagram


#Exercise 3: Frequent patterns and association rules mining

### 0. Load the data file: foodmart.csv


*  A record is a tuple of binary values {0, 1}, each of which denotes the presence of an item (1: bought, 0: not bought).



In [7]:
!git clone https://github.com/nnthaofit/CSC14118.git
df = spark.read.csv("CSC14118/foodmart.csv", header=True, inferSchema = True)
df.show()

fatal: destination path 'CSC14118' already exists and is not an empty directory.


25/04/18 14:45:53 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+------------+---------+-------+--------------+------+---------+----+-------+-------+------------+-----------------+------+------+-----+---------+---------------+-----+--------+------+-------------+------------------+-----------+-------+-----------+--------------+--------+----------+-----------+-----------+----+------+-----------+----------+----+-----------------+---------------+------------+-------------+----------+--------------+-----------------+---+---------+----------+--------------+--------+---------+---------+---+-----+-----+----------+----+----+---------+-------+------------+----+-------+-----------+--------+------------+-----------+-----+-------------+----------------+-----+----------------+-------+---------+------------+-------------+-------------+---------+--------+----+--------+------+------------+-------+---------+------+------------+----+----+----------+------+-------+----------------+-----+----------+----+--------------+-----+------------+----+---------+-------+----+----

### 1. Convert the given data to the format required by Spark MLlib FPGrowth.

In [10]:
from pyspark.sql import functions as F

# Add an 'id' column to the DataFrame
df_with_id = df.withColumn("id", F.monotonically_increasing_id())

# Convert the DataFrame to the required format
formatted_df = df_with_id.withColumn(
    "items",
    F.array(
        *[
            F.when(F.col(column) == 1, F.lit(column)).otherwise(F.lit(None))
            for column in df_with_id.columns if column != "id"
        ]
    )
).withColumn("items", F.expr("filter(items, x -> x is not null)"))

# Select only the id and items columns
formatted_df = formatted_df.select("id", "items")

# Show the formatted DataFrame
formatted_df.show(truncate=False)

+---+---------------------------------------------------------------------+
|id |items                                                                |
+---+---------------------------------------------------------------------+
|0  |[Acetominifen, Cheese, Home Magazines, Shampoo]                      |
|1  |[Acetominifen, Cheese, Hard Candy, Milk, Pot Scrubbers, Rice]        |
|2  |[Coffee, Deli Salads]                                                |
|3  |[Eggs, Gum, Milk, Soup]                                              |
|4  |[Cheese, Dried Fruit, Frozen Chicken, Plastic Utensils]              |
|5  |[Shampoo]                                                            |
|6  |[Milk, Paper Wipes, Waffles]                                         |
|7  |[Donuts, Dried Fruit, Frozen Chicken]                                |
|8  |[Cooking Oil, Hamburger, Maps, Popsicles]                            |
|9  |[Cheese, Cooking Oil, Dips, Preserves, TV Dinner]                    |
|10 |[Nasal 

### 2 Apply Spark MLlib FPGrowth to the formatted data. Mine the set of frequent patterns with the minimum support of 0.1. Mine the set of association rules with the minimum confidence of 0.9.

In [11]:
from pyspark.ml.fpm import FPGrowth
# Create an FPGrowth instance   
# with minSupport and minConfidence set to 0.1 and 0.9


fpGrowth = FPGrowth(
    itemsCol="items",
    minSupport=0.1,
    minConfidence=0.9,
)

# Fit the model
model = fpGrowth.fit(formatted_df)
# Display frequent itemsets
model.freqItemsets.show(truncate=False)
# Display generated association rules
model.associationRules.show(truncate=False)


+-------------+----+
|items        |freq|
+-------------+----+
|[Dried Fruit]|256 |
|[Soup]       |280 |
|[Cookies]    |238 |
|[Cheese]     |285 |
+-------------+----+

+----------+----------+----------+----+-------+
|antecedent|consequent|confidence|lift|support|
+----------+----------+----------+----+-------+
+----------+----------+----------+----+-------+



# Exercise 4: Classification

### 0. Load the data file: mushroom.csv
*   The data represents a collection of mushroom species.
*   There are 8124 examples, each of which has 22 attributes and it is categorized into either “edible” (e) or “poisonous” (p)


In [13]:
df_mushroom = spark.read.csv("CSC14118/mushrooms.csv", header=True, inferSchema = True)
df_mushroom.show()

+-----+---------+-----------+---------+-------+----+---------------+------------+---------+----------+-----------+----------+------------------------+------------------------+----------------------+----------------------+---------+----------+-----------+---------+-----------------+----------+-------+
|class|cap-shape|cap-surface|cap-color|bruises|odor|gill-attachment|gill-spacing|gill-size|gill-color|stalk-shape|stalk-root|stalk-surface-above-ring|stalk-surface-below-ring|stalk-color-above-ring|stalk-color-below-ring|veil-type|veil-color|ring-number|ring-type|spore-print-color|population|habitat|
+-----+---------+-----------+---------+-------+----+---------------+------------+---------+----------+-----------+----------+------------------------+------------------------+----------------------+----------------------+---------+----------+-----------+---------+-----------------+----------+-------+
|    p|        x|          s|        n|      t|   p|              f|           c|        n|   

### 1.	Prepare the train and test sets following the ratio 8:2

In [ ]:
# prepare with train:test = 8:2
train, test = df_mushroom.randomSplit([0.8, 0.2])
train.show()

+-----+---------+-----------+---------+-------+----+---------------+------------+---------+----------+-----------+----------+------------------------+------------------------+----------------------+----------------------+---------+----------+-----------+---------+-----------------+----------+-------+
|class|cap-shape|cap-surface|cap-color|bruises|odor|gill-attachment|gill-spacing|gill-size|gill-color|stalk-shape|stalk-root|stalk-surface-above-ring|stalk-surface-below-ring|stalk-color-above-ring|stalk-color-below-ring|veil-type|veil-color|ring-number|ring-type|spore-print-color|population|habitat|
+-----+---------+-----------+---------+-------+----+---------------+------------+---------+----------+-----------+----------+------------------------+------------------------+----------------------+----------------------+---------+----------+-----------+---------+-----------------+----------+-------+
|    e|        b|          f|        g|      f|   n|              f|           w|        b|   

### 2. Fit a decision tree model on the training set, using Spark MLlib DecisionTreeClassifier with default parameters

### 3. Fit a random forest model on the training set, using Spark MLlib RandomForestClassification with default parameters

### 4. Evaluate the two models on the same test set using the following metrics: areaUnderROC and areaUnderPR

### 5. Chain the above steps into a single pipeline

# Exercise 5: Clustering

### 1.	Cluster the data by using Spark MLlib KMeans with k = 2, 3, and 5, using Euclidean distance and cosine distance

### 2. Evaluate each of the above clustering results using silhoutte score. Which configuration yeilds the best clustering?

### 3. Chain the above steps into a single pipeline

### 4. For each clustering result obtained above, count the number of examples that belong to each of the three species.

## Exercise 6: Network manipulation with Spark GraphFrames

### 0. Load the data files: users.txt and followers.txt

### 1.	Construct a graph from the given data to demonstrate a tiny social network


### 2.	Apply Graphs graphPageRank to the network to obtain a ranking list of users in terms of followers

### 3. Find connected components on the graph, using Graphs connectedComponents or stronglyConnectedComponents